# BirdCLEF 2026 Inference v9 — Fast 3-Fold Ensemble

Loads 9 checkpoints (3 folds × 3 architectures) trained by `birdclef2026-train-v9-fast-ensemble.ipynb`.
Mel parameters are identical to v9 training — do NOT change them.

**Before running:** attach your trained checkpoint dataset as `birdclef-2026-v9-model`
(or the checkpoints will be loaded from `/kaggle/working/` if training ran in the same session).

In [ ]:
# === CELL 1: IMPORTS & CONFIG (must match training exactly) ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)

import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import timm
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── CONFIG — must match v9 training ──────────────────────────────────────────
CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 128,
    n_fft         = 2048,
    hop           = 320,
    fmin          = 50,
    fmax          = 14000,
    architectures = ["resnet50", "efficientnet_b0", "efficientnet_b2"],
    folds         = 3,            # v9: 3 folds
    tta_offsets   = [-1.25, 0.0, 1.25],  # seconds; 3-crop TTA
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)

device = torch.device(CFG["device"])
print("✅ v9 inference config ready")
print(f"   Device     : {device}")
print(f"   Models     : {len(CFG['architectures'])} archs × {CFG['folds']} folds = {len(CFG['architectures'])*CFG['folds']}")
print(f"   TTA crops  : {len(CFG['tta_offsets'])}")

In [ ]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV  = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
TEST_AUDIO    = _first_existing(
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
)
SAMPLE_SUB    = _first_existing(
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
)
# Checkpoint dataset (upload after training, or use /kaggle/working/ if same session)
CKPT_DIR      = _first_existing(
    "/kaggle/input/birdclef-2026-v9-model",
    "/kaggle/working",
)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f"✅ {n_classes} species loaded from taxonomy.csv")
print(f"   TEST_AUDIO : {TEST_AUDIO}")
print(f"   CKPT_DIR   : {CKPT_DIR}")

In [ ]:
# === CELL 3: MEL HELPER (identical params to v9 training) ===
def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ logmel_from_wave defined")
print(f"   n_mels={CFG['n_mels']}, n_fft={CFG['n_fft']}, fmax={CFG['fmax']}Hz")

In [ ]:
# === CELL 4: MODEL ARCHITECTURE (pretrained=False for inference) ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.arch = arch

        if arch == "resnet50":
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features

        else:
            raise ValueError(f"Unsupported arch: {arch}")

        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

print("✅ BirdCLEFModel defined")

In [ ]:
# === CELL 5: LOAD 9 CHECKPOINTS (3 folds × 3 architectures) ===
models = []
missing = []

for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        ckpt_path = Path(CKPT_DIR) / f"{arch}_v9_fold{fold_idx}.pt"
        if not ckpt_path.exists():
            missing.append(str(ckpt_path))
            continue
        m = BirdCLEFModel(arch, n_classes=n_classes, pretrained=False).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
        m.eval()
        models.append(m)
        print(f"   ✅ Loaded {ckpt_path.name}")

if missing:
    print(f"\n⚠️  {len(missing)} checkpoint(s) not found:")
    for p in missing:
        print(f"      {p}")

print(f"\n✅ {len(models)}/{len(CFG['architectures'])*CFG['folds']} models loaded")

In [ ]:
# === CELL 6: PREDICT WITH REAL TTA ===
_use_amp   = (device.type == "cuda")
win_samples = CFG["sr"] * CFG["seconds"]
win_frames  = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1


def predict_recording(audio_path: str, end_second: float) -> np.ndarray:
    """
    Returns averaged sigmoid probabilities (n_classes,) for the window ending at
    end_second, with TTA across CFG['tta_offsets'].
    Returns 0.5 (neutral) if no models are loaded or audio cannot be read.
    """
    if not models:
        return np.full(n_classes, 0.5, dtype=np.float32)

    try:
        wave, sr0 = sf.read(audio_path, always_2d=False)
    except Exception:
        return np.full(n_classes, 0.5, dtype=np.float32)

    if sr0 != CFG["sr"]:
        wave = librosa.resample(wave, orig_sr=sr0, target_sr=CFG["sr"])
    if wave.ndim == 2:
        wave = wave.mean(axis=1)
    wave = wave.astype(np.float32)

    # Pad to ensure all TTA crops are in-bounds
    pad_samples = win_samples + int(max(abs(o) for o in CFG["tta_offsets"]) * CFG["sr"]) + 1
    wave        = np.pad(wave, (pad_samples, pad_samples))

    nom_end   = int(end_second * CFG["sr"]) + pad_samples  # nominal end sample
    nom_start = nom_end - win_samples

    # Build batch of TTA crops
    crops = []
    for offset_s in CFG["tta_offsets"]:
        offset_samp = int(offset_s * CFG["sr"])
        start = max(0, nom_start + offset_samp)
        end   = start + win_samples
        if end > len(wave):
            end   = len(wave)
            start = max(0, end - win_samples)
        clip = wave[start:end]
        if len(clip) < win_samples:
            clip = np.pad(clip, (0, win_samples - len(clip)))
        mel = logmel_from_wave(clip, CFG["sr"])
        # Crop/pad time axis to win_frames
        T = mel.shape[1]
        if T < win_frames:
            mel = np.concatenate([mel, np.zeros((mel.shape[0], win_frames - T), dtype=np.float32)], axis=1)
        elif T > win_frames:
            mel = mel[:, :win_frames]
        crops.append(mel)

    batch = torch.from_numpy(np.stack(crops)[:, None]).float().to(device)  # (TTA, 1, n_mels, W)

    all_probs = []
    for m in models:
        with torch.no_grad(), autocast(enabled=_use_amp):
            probs = torch.sigmoid(m(batch)).float().cpu().numpy()  # (TTA, n_classes)
        all_probs.append(probs.mean(axis=0))                       # average TTA crops

    if not all_probs:
        return np.full(n_classes, 0.5, dtype=np.float32)
    return np.mean(all_probs, axis=0).astype(np.float32)  # average across all models


print("Predict_recording defined")
status = f"{len(models)} models" if models else "WARNING: 0 models loaded - will output neutral 0.5"
print(f"   TTA offsets : {CFG['tta_offsets']} s  ({len(CFG['tta_offsets'])} crops)")
print(f"   Models      : {status}")

In [ ]:
# === CELL 7: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f"Sample submission rows: {len(sample_sub)}")
print(f"Columns: {list(sample_sub.columns[:5])} ...")

results       = []
missing_audio = 0

# Derive soundscape_id as a column to avoid index-misalignment from chained groupby
sample_sub = sample_sub.copy()
sample_sub["_soundscape_id"] = sample_sub["row_id"].str.rsplit("_", n=1).str[0]

for soundscape_id, group in tqdm(
    sample_sub.groupby("_soundscape_id"),
    desc="Soundscapes",
):
    audio_path = None
    for ext in [".ogg", ".wav", ".flac"]:
        cand = Path(TEST_AUDIO) / f"{soundscape_id}{ext}"
        if cand.exists():
            audio_path = str(cand)
            break

    if audio_path is None:
        # Audio not found — emit neutral 0.5 rather than silently skipping
        missing_audio += 1
        for _, row in group.iterrows():
            results.append({"row_id": str(row["row_id"]), **{sp: 0.5 for sp in species}})
        continue

    for _, row in group.iterrows():
        row_id     = str(row["row_id"])
        end_second = int(row_id.rsplit("_", 1)[-1])
        probs      = predict_recording(audio_path, end_second)
        results.append({"row_id": row_id, **{sp: float(probs[i]) for i, sp in enumerate(species)}})

if missing_audio:
    print(f"\u26a0\ufe0f  {missing_audio} soundscape(s) not found in {TEST_AUDIO} \u2014 used neutral 0.5")
print(f"\n\u2705 Generated {len(results)} prediction rows")

In [ ]:
# === CELL 8: BUILD SUBMISSION ===
submission_df = sample_sub[["row_id"]].copy()
submission_df["row_id"] = submission_df["row_id"].astype(str)

if results:
    pred_df = pd.DataFrame(results)
    pred_df["row_id"] = pred_df["row_id"].astype(str)
    submission_df = submission_df.merge(pred_df, on="row_id", how="left")
else:
    print("\u26a0\ufe0f  results list is empty \u2014 building fully neutral submission (0.5)")

# Fill any missing species columns (left-join NaN or fully absent)
for sp in species:
    if sp in submission_df.columns:
        submission_df[sp] = submission_df[sp].fillna(0.5).astype(np.float64)
    else:
        submission_df[sp] = 0.5

submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\u2705 Saved submission.csv  ({len(submission_df)} rows \u00d7 {len(submission_df.columns)} columns)")
submission_df.head(3)

In [ ]:
# === CELL 9: FORMAT VALIDATION ===
print("Running submission validation...")
issues = []

# Check row_id dtype
if submission_df["row_id"].dtype != object:
    issues.append(f"row_id dtype is {submission_df['row_id'].dtype}, expected str")

# Check species columns exist and are numeric
missing_cols = [sp for sp in species if sp not in submission_df.columns]
if missing_cols:
    issues.append(f"{len(missing_cols)} species columns missing: {missing_cols[:5]}")

# Check value range
sp_cols  = [c for c in submission_df.columns if c != "row_id"]
val_min  = submission_df[sp_cols].min().min()
val_max  = submission_df[sp_cols].max().max()
if val_min < 0 or val_max > 1:
    issues.append(f"Predictions out of [0,1]: min={val_min:.4f} max={val_max:.4f}")

# Check for NaN
nan_count = submission_df[sp_cols].isna().sum().sum()
if nan_count > 0:
    issues.append(f"{nan_count} NaN values in species columns")

if issues:
    print("\n⚠️  Issues found:")
    for issue in issues:
        print(f"   • {issue}")
else:
    print("✅ Submission looks valid!")
    print(f"   Rows        : {len(submission_df)}")
    print(f"   Species cols: {len(sp_cols)}")
    print(f"   Value range : [{val_min:.4f}, {val_max:.4f}]")
    print(f"   NaN count   : {nan_count}")